# S6.P2 · 盈亏平衡与信息价值

**问题**：在需求方给出真实参数之前，「做不做联邦、做哪种」取决于一组未知数。哪个未知数最值得先问清楚？

本 notebook 把三个选项写成项目期净值——**不做**（现状，净值恒为 0）、**L1**（本地模型 + 对方 k-匿名聚合统计）、
**L3**（双向联邦）——并对每个未知参数算部分完全信息期望价值（EVPPI）：
如果先把这个参数问清楚再决策，期望净值能提高多少。EVPPI 高的参数排在需求方问题单和后续实验的前面。

- 逻辑在 `modules/m6_evaluation/components/breakeven.py`，全部参数分布在 `modules/m6_evaluation/configs/s6_p2_breakeven.yaml`，逐项注明来源。
- **全部参数都是显式假设**，不含任何业务数据，也不把公开数据集上的命中率差换算成收益。
  本步的产出是「先问哪个数」的顺序与盈亏平衡点的形状，**不是收益结论**。需求方给出实际值后按新值重跑。
- 合规路径按 S0.6R 的规则选取；⚠️ 技术侧自评，未经法务复核（W-001）。

In [1]:
import hashlib
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
ROUND_RATE = 3     # 概率与万港元的显示精度
ROUND_WAN = 2      # 分档 EVPPI 的显示精度（万港元）
ROUND_HIT = 4      # 命中提升的显示精度

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m6_evaluation/configs/s6_p2_breakeven.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()[:HASH_PREFIX_LEN]
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True,
                                  cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + config_hash)
print("seed:", cfg["seed"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", cfg["step_id"], "|", cfg["purpose"])
rng = np.random.default_rng(cfg["seed"])
RESULTS = REPO_ROOT / "modules/m6_evaluation/results"

config: modules/m6_evaluation/configs/s6_p2_breakeven.yaml sha256:649409fccf6f
seed: 20261025
git: 752b9fa（工作区含本步未提交改动）
step: S6.P2 | 在显式假设的参数分布下，比较不做、L1、L3 三个选项的项目期净值，求 L3 的盈亏平衡命中提升，并按部分完全信息期望价值（EVPPI）排出待问参数的优先级


In [2]:
from modules.m6_evaluation.components import breakeven as BE

WAN = 10_000   # 显示单位：万港元
p = BE.sample_parameters(cfg["parameters"], cfg["n_samples"], rng)
p["noise_reference"] = BE.sample_parameter(cfg["noise_reference"], cfg["n_samples"], rng)
p["l3_heavy_path"] = BE.l3_heavy_path(p).astype(float)
npv = BE.npv_options(p)
summary = pd.DataFrame(BE.decision_summary(npv))
summary["evpi"] = BE.evpi(npv)
summary.to_csv(RESULTS / "s6p2_decision_summary.csv", index=False)
view = summary.copy()
view["expected_npv"] = view["expected_npv"] / WAN
view["median_npv"] = view["median_npv"] / WAN
view = view[["option", "expected_npv", "median_npv", "prob_npv_positive", "prob_best"]]
view.columns = ["选项", "期望净值(万港元)", "净值中位数(万港元)", "净值 > 0 的概率", "胜出概率"]
print("抽样 %d 组参数：" % cfg["n_samples"])
print(view.round(ROUND_RATE).to_string(index=False))
print("完全信息期望价值 EVPI ≈ %.1f 万港元" % (BE.evpi(npv) / WAN))
print("L3 须走数据出境安全评估的抽样占比：%.3f" % p["l3_heavy_path"].mean())

抽样 200000 组参数：
选项  期望净值(万港元)  净值中位数(万港元)  净值 > 0 的概率  胜出概率
不做      0.000       0.000       0.000 0.702
L1    -15.133     -61.144       0.260 0.156
L3   -244.081    -296.810       0.162 0.142
完全信息期望价值 EVPI ≈ 132.4 万港元
L3 须走数据出境安全评估的抽样占比：0.626


**读法**：

- 在这组假设下，「不做」胜出约七成；L3 的期望净值约 −244 万港元，净值为正的概率约 0.16；L1 的期望净值约 −15 万港元，接近持平。
- 这**不是**「联邦不划算」的结论：它只说明在需求方给出规模之前，按现有宽泛假设平均下来不划算。
- 完全信息期望价值 EVPI 约 132 万港元，是「所有参数都问清楚」最多值多少钱，也是问题单与试点的预算上限参照。

In [3]:
names = list(cfg["parameters"]) + ["l3_heavy_path", "noise_reference"]
names = [n for n in names if cfg["parameters"].get(n, {}).get("dist") != "constant"]
table = pd.DataFrame(BE.evppi_table(npv, p, names, cfg["n_bins"]))
table["share_of_evpi"] = table["evppi"] / BE.evpi(npv)
table.to_csv(RESULTS / "s6p2_evppi.csv", index=False)
view = table.copy()
view["evppi"] = view["evppi"] / WAN
view.columns = ["参数", "EVPPI(万港元)", "占 EVPI 比例"]
print(view.round(ROUND_RATE).to_string(index=False))

# 推翻条件：两半样本上前 k 名是否一致
half = cfg["n_samples"] // cfg["n_split_check"]
k = cfg["top_k_stability"]
tops = []
for part in range(cfg["n_split_check"]):
    sl = slice(part * half, (part + 1) * half)
    ps = {key: val[sl] for key, val in p.items()}
    ns = {key: val[sl] for key, val in npv.items()}
    tops.append([r["parameter"] for r in BE.evppi_table(ns, ps, names, cfg["n_bins"])[:k]])
print()
for part, top in enumerate(tops):
    print("第 %d 半样本前 %d 名：%s" % (part + 1, k, "、".join(top)))
stable = all(t == tops[0] for t in tops)
print("推翻条件（前 %d 名不一致）是否触发：%s" % (k, "否" if stable else "是"))

                     参数  EVPPI(万港元)  占 EVPI 比例
      contacts_per_year      48.244      0.365
           delta_hit_l3      38.761      0.293
       l1_capture_ratio      17.850      0.135
           causal_share      12.237      0.092
          l1_build_cost       7.892      0.060
          l1_annual_ops       4.870      0.037
     value_per_customer       3.764      0.028
                is_ciio       0.000      0.000
           gba_eligible       0.000      0.000
  training_cohort_small       0.000      0.000
        assessment_cost       0.000      0.000
assessment_delay_months       0.000      0.000
    assessment_approved       0.000      0.000
          contract_cost       0.000      0.000
  contract_delay_months       0.000      0.000
          l3_build_cost       0.000      0.000
          l3_annual_ops       0.000      0.000
     l1_compliance_cost       0.000      0.000
        l1_delay_months       0.000      0.000
          l3_heavy_path       0.000      0.000
        noise


第 1 半样本前 3 名：contacts_per_year、delta_hit_l3、l1_capture_ratio
第 2 半样本前 3 名：contacts_per_year、delta_hit_l3、l1_capture_ratio
推翻条件（前 3 名不一致）是否触发：否


**读法**：

- **年联系人数**排第一（约占 EVPI 的 37%），其次是 **L3 的命中提升**（约 29%）、**L1 捕获比例**（约 13%）、**因果占比**（约 9%）。
- 合规相关参数（是否为关键信息基础设施运营者、是否适用大湾区标准合同、训练人群规模、评估是否通过）在全样本上 EVPPI 都是 0：单独问清其中任何一个，都不会改变「不做」的平均决策。原因是在大部分参数组合里规模不够，合规路径轻重都扭转不了净值的符号。
- 噪声基线（不进入模型的参数）EVPPI 为 0，说明分箱估计在这里没有可见的正偏差。
- 两半样本前 3 名一致，排序可用，推翻条件未触发。
- L1 捕获比例排第三，说明 L1 的同等认真实现（任务清单 T6）本身有决策价值，与 DR-MX-002 · D-5 的恢复顺序一致。

In [4]:
edges = cfg["contact_tiers"]
tier_rows, tier_evppi = [], []
for i in range(len(edges) - 1):
    lo, hi = edges[i], edges[i + 1]
    last = i == len(edges) - 2
    mask = (p["contacts_per_year"] >= lo) & ((p["contacts_per_year"] <= hi) if last
                                           else (p["contacts_per_year"] < hi))
    ps = {key: val[mask] for key, val in p.items()}
    ns = {key: val[mask] for key, val in npv.items()}
    label = "%d–%d" % (lo, hi)
    for r in BE.decision_summary(ns):
        tier_rows.append({"年联系人数": label, "选项": r["option"],
                          "期望净值(万港元)": r["expected_npv"] / WAN, "胜出概率": r["prob_best"]})
    noise = BE.evppi(ns, ps["noise_reference"], cfg["n_bins"]) / WAN
    for r in BE.evppi_table(ns, ps, names, cfg["n_bins"])[:cfg["top_k_evppi"]]:
        tier_evppi.append({"年联系人数": label, "参数": r["parameter"],
                           "EVPPI(万港元)": r["evppi"] / WAN, "该档 EVPI(万港元)": BE.evpi(ns) / WAN,
                           "该档噪声基线(万港元)": noise})
tier_summary = pd.DataFrame(tier_rows)
tier_top = pd.DataFrame(tier_evppi)
tier_top.to_csv(RESULTS / "s6p2_evppi_by_contact_tier.csv", index=False)
print(tier_summary.round(ROUND_RATE).to_string(index=False))
print()
print(tier_top.round(ROUND_WAN).to_string(index=False))

       年联系人数 选项  期望净值(万港元)  胜出概率
  5000–20000 不做      0.000 0.910
  5000–20000 L1    -86.435 0.067
  5000–20000 L3   -461.850 0.023
 20000–50000 不做      0.000 0.637
 20000–50000 L1    -11.309 0.203
 20000–50000 L3   -233.308 0.159
50000–100000 不做      0.000 0.372
50000–100000 L1    122.702 0.272
50000–100000 L3    178.089 0.356

       年联系人数                 参数  EVPPI(万港元)  该档 EVPI(万港元)  该档噪声基线(万港元)
  5000–20000  contacts_per_year        0.00          8.31          0.0
  5000–20000       delta_hit_l3        0.00          8.31          0.0
  5000–20000       causal_share        0.00          8.31          0.0
  5000–20000 value_per_customer        0.00          8.31          0.0
  5000–20000   l1_capture_ratio        0.00          8.31          0.0
  5000–20000            is_ciio        0.00          8.31          0.0
 20000–50000       delta_hit_l3       43.18        105.09          0.0
 20000–50000   l1_capture_ratio       20.08        105.09          0.0
 20000–50000       causal_shar

**读法**：按年联系人数分三档，结论差别很大。

- **不足 2 万**：「不做」胜出约九成，所有参数的 EVPPI 都是 0——再问别的都不改变决策。规模先天不足时，项目应在这里停下。
- **2 万–5 万**：决策取决于 L3 的命中提升，其次是 L1 捕获比例和因果占比；合规参数仍不进前六。
- **5 万以上**：L3 的期望净值转正（约 178 万港元），L1 约 123 万港元。此时**出境机制开始要紧**：「L3 是否须申报数据出境安全评估」的 EVPPI 约 58 万港元，其中是否为关键信息基础设施运营者约 46 万港元。
- 三档的噪声基线都是 0。

含义：需求方问题单的顺序应是**先问规模**（共同客户数与可联系人数），规模过 5 万再问是否为关键信息基础设施运营者与签约主体注册地。规模不足 2 万时，合规问题不必再追。

In [5]:
be = BE.breakeven_delta_hit(p)
finite = np.isfinite(be)
qs = cfg["breakeven_quantiles"]
qrow = pd.DataFrame({"分位": qs, "盈亏平衡命中提升": np.quantile(be[finite], qs)})
print("L3 上线时的盈亏平衡命中提升（绝对值，相对本地单边模型）在参数不确定下的分布：")
print(qrow.round(ROUND_HIT).to_string(index=False))
print("假设的命中提升 ≥ 盈亏平衡点的概率：%.3f" % (p["delta_hit_l3"] >= be).mean())

grid = pd.DataFrame(BE.breakeven_grid(cfg["parameters"], cfg["grid_contacts"], cfg["grid_values"], False)
                    + BE.breakeven_grid(cfg["parameters"], cfg["grid_contacts"], cfg["grid_values"], True))
grid.to_csv(RESULTS / "s6p2_breakeven_grid.csv", index=False)
pivot = grid.pivot_table(index=["compliance_path", "contacts_per_year"], columns="value_per_customer",
                         values="breakeven_delta_hit")
pivot.columns = ["单客 %d 港元" % c for c in pivot.columns]
print()
print("其余参数取中心值时的盈亏平衡命中提升：")
print(pivot.round(ROUND_HIT).to_string())

L3 上线时的盈亏平衡命中提升（绝对值，相对本地单边模型）在参数不确定下的分布：
  分位  盈亏平衡命中提升
0.10    0.0062
0.25    0.0117
0.50    0.0262
0.75    0.0583
0.90    0.1111
假设的命中提升 ≥ 盈亏平衡点的概率：0.192

其余参数取中心值时的盈亏平衡命中提升：
                                    单客 10000 港元  单客 14000 港元  单客 22000 港元
compliance_path  contacts_per_year                                       
个人信息出境标准合同（含大湾区） 5000.0                  0.1068       0.0763       0.0485
                 10000.0                 0.0534       0.0381       0.0243
                 20000.0                 0.0267       0.0191       0.0121
                 50000.0                 0.0107       0.0076       0.0049
                 100000.0                0.0053       0.0038       0.0024
数据出境安全评估         5000.0                  0.1505       0.1075       0.0684
                 10000.0                 0.0753       0.0538       0.0342
                 20000.0                 0.0376       0.0269       0.0171
                 50000.0                 0.0151       0.0108       0.0068
         

**读法**：

- 在参数不确定下，L3 上线所需的命中提升中位数约 2.6 个百分点，四分位区间约 1.2–5.8 个百分点；按假设的命中提升分布，跨过盈亏平衡点的概率约 0.19。
- 确定性表是 10/30 面聊可直接用的形式：其余参数取中心值，**年联系 2 万人、单客 1.4 万港元**时，订立个人信息出境标准合同需约 1.9 个百分点，申报数据出境安全评估需约 2.7 个百分点；年联系 5 万人时分别降到约 0.8 和 1.1 个百分点。
- 数据出境安全评估下的门槛约为个人信息出境标准合同下的 1.4 倍，来自更高的一次性成本和 6–12 个月的上线延迟（经验口径，非法定时限）。
- 表中的命中提升是**绝对值**、相对本地单边模型、在联系名单上计；它只能由真实数据上的随机分组试点测量，公开数据集上的结果不能代入。

## 结论

1. **先问规模。** 年联系人数的 EVPPI 最高；不足 2 万时其余参数都不改变决策，项目应停下。规模过 5 万后，合规路径（是否为关键信息基础设施运营者）才开始影响决策。据此调整需求方问题单顺序（任务清单 T3）。
2. **试点要测两样东西**：命中提升与因果占比，它们在中高规模档的 EVPPI 合计超过 EVPI 的一半。只做离线 AUROC 比较回答不了这两个数。
3. **L1 值得同等认真做**：L1 捕获比例排第三；规模过 5 万时 L1 的期望净值约为 L3 的七成，而合规负担轻得多。
4. 盈亏平衡表给出 10/30 可用的回答形式：「年联系 N 人、单客 V 港元时，联邦方案至少要带来 X 个百分点的命中提升。」

> 全部参数为显式假设（见配置文件逐项注释），结论只是「先问哪个数」的顺序与门槛的形状，不是收益判断。需求方给出实际值后按新值重跑本 notebook。合规路径为技术侧自评，未经法务复核（W-001）。